# Kiểm tra dữ liệu Neo4j (GraphDB)
Notebook này giúp bạn kiểm tra kết nối, xem thống kê số lượng Node/Relationship và thực hiện thử các truy vấn Cypher.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_neo4j import Neo4jGraph

load_dotenv()

# Khởi tạo kết nối, Langchain sẽ tự lấy NEO4J_URI từ biến môi trường
graph = Neo4jGraph()
print("✅ Kết nối Neo4j thành công!")
print("\n=== SCHEMA CỦA GRAPH ===")
print(graph.schema)

## 1. Thống kê số lượng dữ liệu
Đếm tổng số Node và Relationship hiện có trong Graph.

In [ ]:
node_count = graph.query("MATCH (n) RETURN count(n) AS count")[0]["count"]
rel_count = graph.query("MATCH ()-[r]->() RETURN count(r) AS count")[0]["count"]

print(f"Tổng số Nodes: {node_count}")
print(f"Tổng số Relationships: {rel_count}")

## 2. Truy vấn thử một số Node
Lấy thử 5 Node (và label của chúng) để xem dữ liệu có những gì.

In [ ]:
sample_nodes = graph.query("""
MATCH (n)
RETURN labels(n) as labels, properties(n) as props
LIMIT 5
""")

for i, node in enumerate(sample_nodes):
    print(f"\n--- Node {i+1} ---")
    print(f"Labels: {node['labels']}")
    print(f"Properties: {node['props']}")

## 3. Truy vấn tìm một Mối quan hệ cụ thể (Relationship)
Lấy thử 5 mối quan hệ hiện có giữa các Node.

In [ ]:
sample_rels = graph.query("""
MATCH (n)-[r]->(m) 
RETURN labels(n) as n_labels, type(r) as relationship, labels(m) as m_labels
LIMIT 5
""")

for i, rel in enumerate(sample_rels):
    print(f"[{rel['n_labels']}] --({rel['relationship']})--> [{rel['m_labels']}]")

## 4. Kiểm thử Text2Cypher (Tạo mảng Graph RAG)
Dùng Text2Cypher của mô hình OpenAI để tự sinh câu query Graph.

In [ ]:
from langchain.chains import GraphQAChain
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(temperature=0, model="gpt-4o-mini")

chain = GraphQAChain.from_llm(
    llm=llm, 
    graph=graph, 
    verbose=True,      # Bật True để xem câu lệnh Cypher xuất ra là gì
    allow_dangerous_requests=True
)

question = "Tôi ở bộ phận mua sắm. Anh trai tôi mở công ty văn phòng phẩm và muốn làm supplier cho công ty. Tôi có được quyết định chọn công ty của anh trai không và phải xin phép ai?"
print(f"Câu hỏi: {question}\n")

response = chain.invoke({"query": question})
print("\nTrả lời:", response["result"])

## 5. Thống Kê GraphDB (Cypher)

In [ ]:
# Khởi tạo Graph
from langchain_community.graphs import Neo4jGraph
graph = Neo4jGraph()

# Xem tất cả các nhãn (Labels)
labels_query = """
MATCH (n)
UNWIND labels(n) as label
RETURN label, count(*) as count
ORDER BY count DESC
"""
print("=== THỐNG KÊ LABELS MỚI NHẤT ===")
for r in graph.query(labels_query):
    print(f"- {r[label]}: {r[count]} nodes")


## 6. Xem Số Lượng Relationships (Cypher)

In [ ]:
# Xem tất cả các loại Mối quan hệ (Relationships)
rels_query = """
MATCH ()-[r]->()
RETURN type(r) as relationship_type, count(*) as count
ORDER BY count DESC
"""
print("=== THỐNG KÊ RELATIONSHIPS ===")
for r in graph.query(rels_query):
    print(f"- {r[relationship_type]}: {r[count]} relationships")


## 7. Truy vấn Quan hệ 2 Node (Cypher)

In [ ]:
# Thử tìm các Node có quan hệ REQUIRES_APPROVAL_FROM
approval_query = """
MATCH (a)-[r:REQUIRES_APPROVAL_FROM]->(b)
RETURN labels(a)[0] AS Entity_A, a.id AS Name_A, 
       type(r) AS Relationship, 
       labels(b)[0] AS Entity_B, b.id AS Name_B
LIMIT 10
"""
print("=== DANH SÁCH DUYỆT ===")
for r in graph.query(approval_query):
    print(f"[{r["Name_A"]}] cần {r["Relationship"]} từ [{r["Name_B"]}]")
